# Tutorial 5, Q3: Icki-Toffee with binary variables

*Original script: `5 Source Files/Python/icki_toffee_cost_binary.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same optimal value (310).*

*Note: this problem has more than one cheapest selection, so the solver may show a different mix from the answer sheet (which shows 3 each of items 0, 1, 4 and 5, x[9] = 4 and x[11] = 6). Any selection that costs 310 and meets the constraints is correct.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Define integer variables
model = ConcreteModel()
model.x = Var(range(12), within=NonNegativeIntegers)
model.xb = Var(range(12), within=Binary)

In [ ]:
# Define the objective function values
cost = [12, 12, 12, 15, 13, 13, 16, 16, 15, 10, 18, 20]

fat = [20, 16, 10, 7, 6, 5, 13, 15, 18, 0, 19, 0]
cals = [100, 100, 80, 50, 60, 40, 80, 90, 80, 145, 120, 120]
weight = [24, 22, 15, 12, 10, 10, 14, 17, 20, 8, 20, 15]

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(cost[i] * model.x[i] for i in range(12)), sense=minimize)

In [ ]:
# Define the inequality constraints
model.weight_min = Constraint(expr=sum(weight[i] * model.x[i] for i in range(12)) >= 320)
model.weight_max = Constraint(expr=sum(weight[i] * model.x[i] for i in range(12)) <= 1.5 * 320)
model.cals_max = Constraint(expr=sum(cals[i] * model.x[i] for i in range(12)) <= 2850)
model.fat_weight_min = Constraint(expr=sum(fat[i] * model.x[i] for i in range(12)) <= 0.45 * sum(weight[i] * model.x[i] for i in range(12)))

In [ ]:
# Ensure that your selection has at least 6 different candies
model.variety = Constraint(expr=sum(model.xb[i] for i in range(12)) >= 6)

In [ ]:
# Ensure your selection includes at least 3 of each of the selected candies.
model.selection_min = ConstraintList()
for i in range(12):
    model.selection_min.add(model.x[i] >= 3 * model.xb[i])

In [ ]:
# Ensure your selection includes at most 6 of each of the selected candies.
model.selection_max = ConstraintList()
for i in range(12):
    model.selection_max.add(model.x[i] <= 6 * model.xb[i])

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()